#CareLoop Clinics — No-Show Guard

##Part 1 — Python & Data Foundations: ingest, validate, clean, enrich

### Task 1.1 — Read the file the foundations way

In [ ]:
import csv
import json

def parse_row(row):
      row['age'] = int(row['age'])
      row['distance_km']= float(row['distance_km'])
      row['fee_inr'] = int(row['fee_inr'])
      row['prior_no_shows'] = int(row['prior_no_shows'])

      return row

count = 0
accepted =[]
rejected = []
with open('appointments.csv',newline='') as appointments:
  reader = csv.DictReader(appointments)
  print("Field Names:")
  print(reader.fieldnames)

  for rows in reader:
    count = count+1
    try:
      converted_rows = parse_row(rows)
      accepted.append(converted_rows)

    except ValueError:
      rejected.append(rows)




  print("Row Count:",count)
  print("Accepted Rows:",len(accepted))
  print("Rejected Rows:",len(rejected))

with open('rejected_rows.json','w') as file:
    json.dump(rejected,file,indent =2)




Field Names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row Count: 1020
Accepted Rows: 988
Rejected Rows: 32


### Task 1.2 — Inspect and clean

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments.csv")

print("Shape of the Dataframe:", df.shape)

print("First 5 Rows")
print("*"*50)
df.head()


Shape of the Dataframe: (1020, 14)
First 5 Rows
**************************************************


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended


In [ ]:
#Find the Information about the Dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB


In [ ]:
#Find the count of null values.
df.isnull().sum()

,0
appointment_id,0
patient_id,0
clinic_id,0
booked_on,0
appt_date,1
age,12
gender,0
distance_km,20
fee_inr,0
booking_channel,0


In [ ]:
#print Duplicate records count
print("Duplicated :",df.duplicated().sum())

Duplicated : 20


In [ ]:
#Drop Duplicate Records
df = df.drop_duplicates()
print("Shape of Dataframe after removing Duplicates",df.shape)

Shape of Dataframe after removing Duplicates (1000, 14)


In [ ]:
print("Count of Unique Values:",df['booking_channel'].nunique())

Count of Unique Values: 16


In [ ]:
#Clean booking channel

df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print("Count of Unique Values:",df['booking_channel'].nunique())
print(df['booking_channel'].value_counts())

Count of Unique Values: 3
booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64


In [ ]:
# convert appt_date with pd.to_datetime(..., errors="coerce") and booked_on with pd.to_datetime

df['appt_date'] = pd.to_datetime(df['appt_date'],errors ="coerce")
df['booked_on'] = pd.to_datetime(df['booked_on'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   appointment_id   1000 non-null   object        
 1   patient_id       1000 non-null   object        
 2   clinic_id        1000 non-null   object        
 3   booked_on        1000 non-null   datetime64[ns]
 4   appt_date        994 non-null    datetime64[ns]
 5   age              988 non-null    float64       
 6   gender           1000 non-null   object        
 7   distance_km      980 non-null    float64       
 8   fee_inr          1000 non-null   int64         
 9   booking_channel  1000 non-null   object        
 10  visit_type       1000 non-null   object        
 11  reminder_sent    1000 non-null   object        
 12  prior_no_shows   1000 non-null   int64         
 13  status           996 non-null    object        
dtypes: datetime64[ns](2), float64(2), int64(2), o

In [ ]:
#drop rows where appt_date is missing after conversion or status is missing
df = df.dropna(subset=["appt_date","status"])
df.shape

(990, 14)

### Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
#Treat any age below 1 or above 100 as invalid and set it to missing

print("Invalid Ages:",((df['age']<1)|(df['age']>100)).sum())

df.loc[((((df['age']<1)|(df['age']>100)))),'age'] = np.nan

print("Missing Ages:",df['age'].isna().sum())

Invalid Ages: 5
Missing Ages: 17


In [ ]:
#Fill missing age with the median (then .round().astype(int))

age_median = df['age'].median()
print("Age Median",age_median)

df['age'] = df['age'].fillna(age_median).round().astype(int)
print("Missing Ages:",df['age'].isna().sum())
print("Age Datatype:", df['age'].dtype)

Age Median 48.0
Missing Ages: 0
Age Datatype: int64


In [ ]:
#Fill missing distance_km with its median.
dist_median = (df['distance_km'].median())
print("Dist_Median:",dist_median)

df['distance_km'] = df['distance_km'].fillna(dist_median)
print("Missing Distance:",df['distance_km'].isna().sum())

Dist_Median: 8.25
Missing Distance: 0


In [ ]:
#For distance_km and fee_inr, compute Q1, Q3 and IQR with .quantile,
#and cap values at Q3 + 1.5×IQR and Q1 − 1.5×IQR with np.clip



print("Rows:", len(df))
print("Q1:", df['distance_km'].quantile(0.25))
print("Q3:", df['distance_km'].quantile(0.75))

distance_q1 = df['distance_km'].quantile(0.25)
distance_q3 = df['distance_km'].quantile(0.75)

IQR_distance = distance_q3 - distance_q1

print("Distance_Q1:",distance_q1)
print("Distance_Q3:",distance_q3)
print("IQR:",IQR_distance)

lower_distance = distance_q1 - 1.5 * IQR_distance
higher_distance = distance_q3 + 1.5 * IQR_distance

print("Lower Limit:",lower_distance )
print("Higher Limit:", higher_distance)

distance_outliers = ((df['distance_km'] < lower_distance)|(df['distance_km']>higher_distance))

print("Distance Outliers:",distance_outliers.sum())

df['distance_km'] = np.clip(df['distance_km'],lower_distance,higher_distance)

print("Minimum Distance Value:", df['distance_km'].min())
print("Maximum Distance Value:", df['distance_km'].max())




Rows: 990
Q1: 4.625
Q3: 13.0
Distance_Q1: 4.625
Distance_Q3: 13.0
IQR: 8.375
Lower Limit: -7.9375
Higher Limit: 25.5625
Distance Outliers: 26
Minimum Distance Value: 0.6
Maximum Distance Value: 25.5625


In [ ]:
#Fee caping

fee_q1 = df['fee_inr'].quantile(0.25)
fee_q3 = df['fee_inr'].quantile(0.75)

fee_iqr = fee_q3 - fee_q1

print("Fee_Q1:",fee_q1)
print("Fee_Q3:",fee_q3)
print("Fee IQR:",fee_iqr)

min_fee = fee_q1 - 1.5*fee_iqr
max_fee = fee_q3 + 1.5*fee_iqr

print("Lower Limit Fee:",min_fee)
print("Higher Limit Fee:", max_fee)

fee_outliers = ((df['fee_inr'] < min_fee)|(df['fee_inr'] > max_fee))

print("Fee Oultiers:",fee_outliers.sum())

df['fee_inr'] = np.clip(df['fee_inr'],min_fee,max_fee)

print("Minimum Fee Value:", df['fee_inr'].min())
print("Maximum Fee Value:", df['fee_inr'].max())



Fee_Q1: 500.0
Fee_Q3: 787.5
Fee IQR: 287.5
Lower Limit Fee: 68.75
Higher Limit Fee: 1218.75
Fee Oultiers: 3
Minimum Fee Value: 300.0
Maximum Fee Value: 1218.75


In [ ]:
# (d) Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int)).


df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

print("Distance in Km rounded to 1 Sample:",df['distance_km'].head(1))
print("Fee Inr Datatype:",df['fee_inr'].dtype)

Distance in Km rounded to 1 Sample: 0    18.0
Name: distance_km, dtype: float64
Fee Inr Datatype: int64


In [ ]:
#Print how many values you capped in each column.
print("No of Values capped in Fees:",fee_outliers.sum())
print("No of Values capped in Distance:",distance_outliers.sum())

No of Values capped in Fees: 3
No of Values capped in Distance: 26


### Task 1.4 — Features and a NumPy summary

Create lead_days = days between booked_on and appt_date (use .dt.days) and no_show = 1 if status is "No-Show" else 0. Then take X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy() and, without loops, compute: (a) the mean of each column (axis=0); (b) the number and share of rows with lead_days > 14 using a boolean mask (mask.sum() counts the True values and mask.mean() gives the share); (c) z-scores of the distance_km column using (x - np.mean(x)) / np.std(x) and the count of rows with |z| > 2.

In [ ]:
#Create lead_days = days between booked_on and appt_date (use .dt.days)
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
print("Minimum Lead Days",df['lead_days'].min())
print("Maximum Lead Days",df['lead_days'].max())

Minimum Lead Days 1
Maximum Lead Days 45


In [ ]:
#no_show = 1 if status is "No-Show" else 0
df['no_show'] = np.where(df['status']=="No-Show",1,0)
print(df['no_show'].value_counts())

no_show
0    772
1    218
Name: count, dtype: int64


In [ ]:
print("No Show Count:",df['no_show'].sum())
print("Total Count:",len(df))

no_show_per = round(((df['no_show'].sum() /len(df)) * 100),1)
#218 no-shows out of 990 (22.0%)
print(f"{df['no_show'].sum()} no-shows out of {len(df)} ({no_show_per}%)")

No Show Count: 218
Total Count: 990
218 no-shows out of 990 (22.0%)


In [ ]:
#take X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy()

X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy()
X.shape

(990, 4)

In [ ]:
#the mean of each column (axis=0)
X_means = np.mean(X,axis = 0)
print("The mean of each column : ",X_means )

The mean of each column :  [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [ ]:
#the number and share of rows with lead_days > 14 using a boolean mask
# (mask.sum() counts the True values and mask.mean() gives the share)

mask = X[:,3] > 14

print("Count of True Values:",mask.sum())
print("The share of True values:",mask.mean())
print("Ther percentage of True Values:",mask.mean()*100)

Count of True Values: 296
The share of True values: 0.298989898989899
Ther percentage of True Values: 29.898989898989896


In [ ]:
# z-scores of the distance_km column using (x - np.mean(x)) / np.std(x) and the count of rows with |z| > 2.

x = X[:,1]

z_score = (x - np.mean(x))/np.std(x)
abs_z_score = np.abs(z_score)>2
print("The count of rows with Z score greater than 2:", abs_z_score.sum())

The count of rows with Z score greater than 2: 53


### Task 1.5 — Enrich the clinics from a live API

For each of the 8 clinics call the free Open-Meteo API with requests.get("https://api.open-meteo.com/v1/forecast", params={"latitude": ..., "longitude": ..., "hourly": "temperature_2m", "forecast_days": 1}). Check status_code == 200 before using the reply, read the JSON, take the first value of ["hourly"]["temperature_2m"] (the midnight-UTC reading, a stand-in rather than the current weather) as temp_c, and save clinics_enriched.csv (the original columns plus temp_c; df.to_csv("file.csv", index=False) writes a DataFrame to CSV). No API key is needed. If the API fails: pass timeout=10 to requests.get, wrap the call in try/except requests.exceptions.RequestException, retry that clinic once, and if it still fails (or the status is not 200, or hourly is missing) store None as temp_c, print the clinic's name and the reason, and carry on. Never type in a made-up temperature; in load_data.sql a missing value is written as NULL (unquoted).

In [ ]:
clinic_df = pd.read_csv("clinics.csv")
print(clinic_df.head())
print(clinic_df.columns)

  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
Index(['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude'], dtype='object')


In [ ]:
#Test for 1 Clinic

import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {"latitude": clinic_df.loc[0,"latitude"],
"longitude":clinic_df.loc[0,"longitude"] ,
"hourly": "temperature_2m",
"forecast_days": 1}

response = requests.get(url,params = params,timeout = 10)

print("Status code:",response.status_code)



Status code: 200


In [ ]:
#Test Data
data = response.json()

print(data["hourly"]["temperature_2m"][0])
temp_c = data["hourly"]["temperature_2m"][0]
print(temp_c)

16.7
16.7


In [ ]:
temp_list = []
for index,row in clinic_df.iterrows():
  latitude = row["latitude"]
  longitude = row["longitude"]
  clinic_name = row["clinic_name"]

  params = {"latitude": latitude,
            "longitude":longitude ,
            "hourly": "temperature_2m",
            "forecast_days": 1}

  temp_c = None

  for attempts in range(2):

      try:
        response = requests.get(url,params = params,timeout = 10)

        if response.status_code!=200:
          reason =f"status_code:{response.status_code}"

        else:
          data = response.json()

          if ("hourly" in data and "temperature_2m" in data["hourly"] and len(data["hourly"]["temperature_2m"])>0):
            temp_c = data["hourly"]["temperature_2m"][0]
            break
          else:
            reason ="Hourly Data Missing"

      except requests.RequestException as e:
        reason = str(e)

      if attempts == 1:
        print(clinic_name,"-",reason )


  temp_list.append(temp_c)

  print(clinic_name,temp_c)


Kothrud Family Clinic 16.7
Hadapsar Care Centre 18.0
Sitabuldi Health Point 20.1
Malviya Nagar Clinic 17.4
Vaishali Nagar Clinic 17.4
Edappally Medical Centre 24.9
Vijay Nagar Clinic 20.1
Gomti Nagar Care Point 16.4


In [ ]:
#Enrich DS
clinic_df['temp_c'] = temp_list

clinic_df.head()

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,16.7
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.0
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.1
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,17.4
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,17.4


In [ ]:
#Export the enriched DS
clinic_df.to_csv("clinics_enriched.csv",index = False)

### Task 1.6 — Export the clean table and write the SQL load script

Sort the cleaned data by appointment_id, format both date columns as YYYY-MM-DD text (.dt.strftime("%Y-%m-%d")), and save appointments_clean.csv with exactly these 16 columns, in this order: appointment_id, patient_id, clinic_id, booked_on, appt_date, age, gender, distance_km, fee_inr, booking_channel, visit_type, reminder_sent, prior_no_shows, status, lead_days, no_show. Then, with a for loop and f-strings, write load_data.sql containing one statement per row, in this pattern (text in single quotes, numbers unquoted): INSERT INTO appointments VALUES ('A0001','P196','C05','2025-01-22',...,0); — and likewise one INSERT INTO clinics VALUES (...) per clinic from clinics_enriched.csv. Write all the clinics statements first, because the appointments table points to them with a foreign key. Before exporting, check and print that appointment_id is unique and that none of the 16 columns has a missing value. (Building SQL text with f-strings is fine for this fixed, controlled dataset — none of its values contains an apostrophe. It is not a safe method for real applications, where parameterised queries are used instead.)

In [ ]:
#Select the required column list
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
#Sort by appointment_id
df = df.sort_values(by ="appointment_id")
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
#Check appointment ID uniqueness
print("Unique Appointment Id",df['appointment_id'].nunique())
print("Total Rows:",len(df))

Unique Appointment Id 990
Total Rows: 990


In [ ]:
#Step 3: Check missing values
df.isnull().sum()

,0
appointment_id,0
patient_id,0
clinic_id,0
booked_on,0
appt_date,0
age,0
gender,0
distance_km,0
fee_inr,0
booking_channel,0


In [ ]:
#Format the date columns.
df['booked_on'] = df['booked_on'].dt.strftime("%Y-%m-%d")
df['appt_date'] = df['appt_date'].dt.strftime("%Y-%m-%d")


In [ ]:
print("Booked on Sample:",df['booked_on'].head(1))
print("Appt Date Sample:",df['appt_date'].head(1))

df.info()

Booked on Sample: 0    2025-01-24
Name: booked_on, dtype: object
Appt Date Sample: 0    2025-01-25
Name: appt_date, dtype: object
<class 'pandas.core.frame.DataFrame'>
Index: 990 entries, 0 to 1019
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   990 non-null    object 
 1   patient_id       990 non-null    object 
 2   clinic_id        990 non-null    object 
 3   booked_on        990 non-null    object 
 4   appt_date        990 non-null    object 
 5   age              990 non-null    int64  
 6   gender           990 non-null    object 
 7   distance_km      990 non-null    float64
 8   fee_inr          990 non-null    int64  
 9   booking_channel  990 non-null    object 
 10  visit_type       990 non-null    object 
 11  reminder_sent    990 non-null    object 
 12  prior_no_shows   990 non-null    int64  
 13  status           990 non-null    object 
 14  lead_days        990 non-nul

In [ ]:
#Export appointments_clean.csv

df.to_csv("appointments_clean.csv",index = False)
print("Successfully exported appointments_clean.csv")

Successfully exported appointments_clean.csv


In [ ]:
clinic_df.head(2)

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,16.7
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.0


In [ ]:
len(clinic_df)

8

In [ ]:
#writing load_data.sql

sql_statements = []

for index, row in clinic_df.iterrows():
  clinic_id = row['clinic_id']
  clinic_name = row['clinic_name']
  city = row['city']
  tier = row['tier']
  latitude = row['latitude']
  longitude = row['longitude']
  if pd.isna(row['temp_c']):
    temperature = "Null"
  else:
    temperature = row['temp_c']

  statement = f"insert into clinics values('{clinic_id}','{clinic_name}','{city}','{tier}',{latitude},{longitude},{temperature});"
  sql_statements.append(statement)

In [ ]:
print(len(sql_statements))
print(sql_statements)

8
["insert into clinics values('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,16.7);", "insert into clinics values('C02','Hadapsar Care Centre','Pune','Metro',18.5089,73.926,18.0);", "insert into clinics values('C03','Sitabuldi Health Point','Nagpur','Tier-2',21.1458,79.0882,20.1);", "insert into clinics values('C04','Malviya Nagar Clinic','Jaipur','Tier-2',26.8553,75.815,17.4);", "insert into clinics values('C05','Vaishali Nagar Clinic','Jaipur','Tier-2',26.9124,75.7873,17.4);", "insert into clinics values('C06','Edappally Medical Centre','Kochi','Tier-2',10.0261,76.3085,24.9);", "insert into clinics values('C07','Vijay Nagar Clinic','Indore','Tier-2',22.7533,75.8937,20.1);", "insert into clinics values('C08','Gomti Nagar Care Point','Lucknow','Tier-2',26.8467,80.9462,16.4);"]


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 990 entries, 0 to 1019
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   990 non-null    object 
 1   patient_id       990 non-null    object 
 2   clinic_id        990 non-null    object 
 3   booked_on        990 non-null    object 
 4   appt_date        990 non-null    object 
 5   age              990 non-null    int64  
 6   gender           990 non-null    object 
 7   distance_km      990 non-null    float64
 8   fee_inr          990 non-null    int64  
 9   booking_channel  990 non-null    object 
 10  visit_type       990 non-null    object 
 11  reminder_sent    990 non-null    object 
 12  prior_no_shows   990 non-null    int64  
 13  status           990 non-null    object 
 14  lead_days        990 non-null    int64  
 15  no_show          990 non-null    int64  
dtypes: float64(1), int64(5), object(10)
memory usage: 131.5+ KB


In [ ]:
for index, row in df.iterrows():
    appointment_id = row['appointment_id']
    patient_id = row['patient_id']
    clinic_id = row['clinic_id']
    booked_on = row['booked_on']
    appt_date = row['appt_date']
    age = row['age']
    gender =row['gender']
    distance_km = row['distance_km']
    fee_inr = row['fee_inr']
    booking_channel = row['booking_channel']
    visit_type = row['visit_type']
    reminder_sent = row['reminder_sent']
    prior_no_shows = row['prior_no_shows']
    status = row['status']
    lead_days = row['lead_days']
    no_show = row['no_show']


    statement = (
    f"INSERT INTO appointments VALUES "
    f"('{appointment_id}', '{patient_id}', '{clinic_id}', "
    f"'{booked_on}', '{appt_date}', {age}, '{gender}', "
    f"{distance_km}, {fee_inr}, '{booking_channel}', "
    f"'{visit_type}', '{reminder_sent}', {prior_no_shows}, "
    f"'{status}', {lead_days}, {no_show});"
)

    sql_statements.append(statement)


In [ ]:
print(len(sql_statements))

998


In [ ]:
#Write everything into load_data.sql
with open("load_data.sql",'w') as file:
  for st in sql_statements:
    file.write(st +"\n")

print("File created")


File created


In [ ]:
#Verify the file
with open("load_data.sql",'r') as file:
  lines = file.readlines()

print("Total insert Statements:", len(lines))
print(lines[0])
print(lines[8])

Total insert Statements: 998
insert into clinics values('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,16.7);

INSERT INTO appointments VALUES ('A0001', 'P509', 'C05', '2025-01-24', '2025-01-25', 60, 'F', 18.0, 850, 'Phone', 'Consultation', 'Yes', 1, 'No-Show', 1, 1);



## Part 2 — SQL: verify the numbers and find where no-shows cluster

### Task 2.2 — No-show rate by clinic

In [ ]:
# Python vs SQL verification
clinic_summary = df.groupby('clinic_id').agg(appointments = ('appointment_id','count'),
     no_shows = ('no_show','sum')
 ).reset_index()

clinic_summary['no_show_pct'] = (100 * ((clinic_summary['no_shows'])/(clinic_summary['appointments']))).round(1)

clinic_summary = clinic_summary.sort_values(by='no_show_pct', ascending = False)

print(clinic_summary)


  clinic_id  appointments  no_shows  no_show_pct
7       C08            90        30         33.3
1       C02           143        39         27.3
3       C04           142        38         26.8
0       C01           160        36         22.5
6       C07           101        21         20.8
5       C06           142        23         16.2
2       C03            82        13         15.9
4       C05           130        18         13.8
